<a href="https://colab.research.google.com/github/DorineEliseVeyrat92/Master-Thesis-Enterprise-Query-Routing/blob/main/Thesis_MiniMock_Version1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Table of content:

* A) Set-up & Configuration
* B) System Definition
* C) Routing Components / Orchestrator
* D) Standard evaluation
* E) Additional experiments


A) Set-up & Configuration

A1) Version Header

ThesisMiniMock version = 1.2.0 (Architecture V1 – Unified Orchestration, Multi-Tool)

Goal: This notebook implements a simplified orchestration layer to evaluate agent routing
strategies for a multi-tool LLM system. The target system is currently in a design phase,
and multiple orchestration architectures are under evaluation.

Architectural options under consideration include:
- V1: Unified orchestration architecture where a single routing layer selects amoung all available tools
- V2: Split orchestration architecture with two specialized routing paths corresponding to operational procurement data and knowledge-oriented information sources

This notebook focuses on Architecture V1, and evaluates a unified orchestration design with improved routing models amd confidence-baed fallback behavior.

Routing strategies evaluated:
- Embedding-based routing
- LLM-based routing
- Hybrid routing (rules → embeddings → LLM fallback)

Governance includes a confidence threshold with HITL escalation and decision trace logging
for explainability.

Run order: top to bottom


2. Imports

In [ ]:
# =============================================
# A2) Imports (Version 1.0.0)
# =============================================

# Standard librairies
import re
import json
import time
from dataclasses import dataclass, field
from typing import Any, Dict, List, Tuple

# Data / numerical
import numpy as np
import pandas as pd

# Embeddings / similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Hugging Face LLM
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline


3. Configuration constants

In [ ]:
# ==============================
# A3) Configuration constants (V1.2)
# ==============================

EXPERIMENT_NAME = "ThesisMiniMock"
ARCHITECTURE = "V1_UNIFIED_ORCHESTRATION"
VERSION = "1.2.0"

# Governance
CONFIDENCE_THRESHOLD = 0.80
ACTION_AUTO_ROUTE = "Auto_Route"
ACTION_HITL = "HITL_Escalation"

USE_FALLBACK = False
FALLBACK_THRESHOLD = 0.55

# Routing Strategies
STRATEGIES = ["Embedding_Based", "LLM_Router", "Hybrid"]
DEFAULT_STRATEGY = "Hybrid"

# Routing models
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"
LLM_MODEL_NAME = "facebook/bart-large-mnli"

# Evaluation / reporting
TOP_K_EMBEDDING_SCORES_TO_SHOW = 2
PRINT_REASONING = True
PRINT_DECISION_TRACE = True

EXPORT_PATH = f"{EXPERIMENT_NAME}_{ARCHITECTURE}_v{VERSION}_results.csv"


In [ ]:
# =============================================
# A4) Load Datasets
# =============================================

df_baseline = pd.read_csv("baseline_dataset.csv")
df_noisy = pd.read_csv("noisy_dataset.csv")
df_paraphrased = pd.read_csv("paraphrased_dataset.csv")
df_translated = pd.read_csv("translated_dataset.csv")

B) System Definition

1. Data Models
2. Tools catalog & Pool defintion
3. Query Preprocessing
4. Candidate Tools Selection

In [ ]:
# ============================================================
# B1) Data Models (Version 1.0.0 - Architecture V1)
# ============================================================


@dataclass
class ToolPool:
    """Tool pool containing all tools available to the orchestrator."""
    name: str
    description: str
    tools: List[Dict[str, str]]
    #metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class SelectionResult:
    """Intermediate model selection output before confidence-based governance is applied."""
    selected_tool: str
    confidence: float
    trace: Dict[str, Any] = field(default_factory=dict)

@dataclass
class FinalResult:
    """Final system output after governance and logging."""
    query_id: int
    query: str
    strategy: str
    decision: SelectionResult
    action: str
    threshold: float


In [ ]:
# ============================================================
# B2) Tool Catalog and Pool Configuration (Version 1.2.0 - Architecture V1)
# Unified orchestration
# ============================================================

# Tool descriptions
GENERIC_DESC = "Answers generic procurement questions without calling structured sources."
GRBW_DESC = "Operational purchase-order line data (transactional)."
OA_DESC = "Outline agreement master data (contract types, validity, vendor, purchasing structure)."
OA_FULFILLMENT_DESC = "Outline agreement fulfillment & consumption KPIs (released/remaining budget, consumption quantities)."
SHAREPOINT_TRAINING_DESC = "Training materials and learning content from SharePoint (topics, responsibilities, sessions)."
ICERTIS_DESC = "Signed contract documents and extracted legal/contract metadata."
EMAIL_SHAREPOINT_DATA_DESC = "Email + SharePoint attachments analysis (pricing, suppliers, incoterms, quantities, etc.)."

TOOLS_ALL = [
    {"name": "grbw_data_tool", "description": GRBW_DESC},
    {"name": "oa_data_tool", "description": OA_DESC},
    {"name": "oa_data_fulfillment_tool", "description": OA_FULFILLMENT_DESC},
    {"name": "sharepoint_training_tool", "description": SHAREPOINT_TRAINING_DESC},
    {"name": "icertis_data_tool", "description": ICERTIS_DESC},
    {"name": "email_sharepoint_data_tool", "description": EMAIL_SHAREPOINT_DATA_DESC},
    {"name": "generic_tool", "description": GENERIC_DESC},
]

TOOL_POOLS: List[ToolPool] = [
    ToolPool(
        name="AllTools",
        description="Unified tool pool with access to all procurement tools.",
        tools=TOOLS_ALL
    )
]


In [ ]:
# ============================
# B3) Query Preprocessing (V1.2)
# ============================

def normalize_text(text: str) -> str:
    """Lowercase, trim, collapse whitespace."""
    return re.sub(r"\s+", " ", text.strip().lower())

def preprocess_query(query: str) -> Tuple[str, Dict[str, Any]]:
    """Main preprocessing entry point before tool selection."""
    return normalize_text(query)


In [ ]:
# ============================
# B4) Candidate Pool Selection (V1.2)
# ============================

def get_candidate_tools_v1(pool: ToolPool) -> List[Dict[str, Any]]:
    """ V1: all tools are always available (no filtering) """
    return pool.tools


C) Routing Components / Orchestrator

1. Router 1 : Embedding based
2. Router 2: LLM-router
3. Hybrid cascade
4. Orchestrator wrapper

In [ ]:
# ============================
# C1) Embedding Router (V1.2)
# ============================

class EmbeddingRouter:
    """
    Scores candidate tools using embedding cosine similarity between:
      - query text
      - tool.description text
    Embedding generated via Sentence Transformer.
    """
    def __init__(self, model_name: str, top_k: int = 3, use_fallback: bool = USE_FALLBACK, fallback_threshold=FALLBACK_THRESHOLD):
        self.top_k = top_k
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self.use_fallback = use_fallback
        self.fallback_threshold = fallback_threshold
        use_fallback: bool = USE_FALLBACK
        fallback_threshold: float = FALLBACK_THRESHOLD

    def route(self, query: str, candidate_tools: List[Dict[str, Any]]) -> SelectionResult:
      #Seperate generic fallback tool only if fallback is activated
      if self.use_fallback:
          specific_tools = [t for t in candidate_tools if t["name"] != "generic_tool"]
          generic_tool = next((t for t in candidate_tools if t["name"] == "generic_tool"), None)
      else:
          specific_tools = candidate_tools
          generic_tool = None

      # Extract tool names and descriptions
      texts = [t.get("description", "") for t in candidate_tools]
      names = [t.get("name", "") for t in candidate_tools]

      # Encode query and candidate tool
      query_vec = self.model.encode([query], convert_to_numpy=True)
      tool_vecs = self.model.encode(texts, convert_to_numpy=True)

      # Cosine simiarity: one score per candidate
      sims = cosine_similarity(query_vec, tool_vecs).flatten()  # one score per tool
      best_idx = int(np.argmax(sims))
      best_name = names[best_idx]
      best_score = float(sims[best_idx])

      #Optional fallback
      if self.use_fallback:
         if best_name == "":
             best_name = generic_tool["name"] if generic_tool else specific_tools[0]["name"]
      elif best_score < self.fallback_threshold and generic_tool is not None:
          selected_tool = generic_tool["name"]


      # Trace: top-K scores
      top_idx = np.argsort(sims)[::-1][: self.top_k]
      top_scores = [(names[i], float(sims[i])) for i in top_idx]

      return SelectionResult(
          selected_tool=best_name,# In V1, we store "selected tool name" here
          confidence=best_score,
          trace={
              "strategy": "embedding",
              "pool": "unified",
              "model": self.model_name,
              "top_scores": top_scores,
              "num_candidates": len(candidate_tools),
            },
        )



In [ ]:
# ============================
# C2) Zero-ShotLLM Router (V1.2)
# ============================

class LLMRouter:

  def __init__(self, model_name: str, use_fallback: bool = USE_FALLBACK, fallback_threshold=FALLBACK_THRESHOLD):
    self.model_name = model_name
    self.use_fallback = use_fallback
    self.fallback_threshold = fallback_threshold
    self.classifier = pipeline(
        "zero-shot-classification",
        model=model_name
    )

  def route(self, query, candidate_tools: List[Dict[str, Any]]):
      #Seperate generic fall back tool exclusion only if fallback is activated
      if self.use_fallback:
          specific_tools = [t for t in candidate_tools if t["name"] != "generic_tool"]
          generic_tool = next((t for t in candidate_tools if t["name"] == "generic_tool"), None)
      else :
          specific_tools = candidate_tools
          generic_tool = None

      #Extract labels from tool descirtption
      labels = [t["description"] for t in specific_tools]

     #Classify query
      result = self.classifier(
          query,
          candidate_labels=labels,
          hypothesis_template="The best tool to answer this query is {}",
         )

      best_label = result["labels"][0]
      confidence = float(result["scores"][0])

      #Map desciption back to tool name
      selected_tool = None
      for t in specific_tools:
         if t["description"] == best_label:
              selected_tool = t["name"]
              break
      #Seperate generic fallback tool only if fallback is activated
      if self.use_fallback:
         if selected_tool is None:
             selected_tool = generic_tool["name"] if generic_tool else specific_tools[0]["name"]
      elif confidence < self.fallback_threshold and generic_tool is not None:
          selected_tool = generic_tool["name"]

      return SelectionResult(
          selected_tool=selected_tool,
          confidence=float(confidence),
          trace={
              "strategy": "llm",
              "pool":"Unified",
              "model": self.model_name,
              "candidate_count":len(candidate_tools),
              "selected_description": best_label,
              "fallback_enabled": self.use_fallback
          }
      )



In [ ]:
## ============================
## C3) Hybrid Router (V1.2)
## ============================

class HybridRouter:
    """
    Hybrid cascade:
      1) embedding router
      2) LLM fallback if embedding confidence is below treshold
    """

    def __init__(self, embedding_router: EmbeddingRouter, llm_router: LLMRouter, threshold: float):
        self.embedding_router = embedding_router
        self.llm_router = llm_router
        self.threshold = threshold

    def route(
        self,
        query: str,
        candidate_tools: List[Dict[str, Any]]
    ) -> SelectionResult:
    #Step 1: Embedding Router
      emb = self.embedding_router.route(query, candidate_tools)
      emb.trace["cascade_stage"] = "embedding"

      if emb.confidence >= self.threshold:
        emb.trace["final_choice"] = "embedding"
        return emb

    #Step 2: LLM fallback
      llm = self.llm_router.route(query, candidate_tools)

      llm.trace["cascade_stage"] = "llm-fallback"
      llm.trace["prev_embedding_conf"] = emb.confidence
      llm.trace["embedding_suggestion"] = emb.selected_tool
      llm.trace["treshold"] = self.threshold

      return llm


In [ ]:
# ============================
# C4) Orchestrator Wrapper (V1.2)
# ============================

class MiniOrchestratorV1_2:
    def __init__(self, tool_pools: List[ToolPool], threshold: float = CONFIDENCE_THRESHOLD):
        self.tool_pools = tool_pools
        self.threshold = threshold

        # V1 expects exactly one unified pool
        self.unified_pool = tool_pools[0]
        self.tools_all = self.unified_pool.tools

        self.embedding_router = EmbeddingRouter(
            model_name=EMBEDDING_MODEL_NAME,
            top_k=TOP_K_EMBEDDING_SCORES_TO_SHOW,
            use_fallback=USE_FALLBACK,
            fallback_threshold=FALLBACK_THRESHOLD
            )
        self.llm_router = LLMRouter(
            model_name=LLM_MODEL_NAME,
            use_fallback=USE_FALLBACK,
            fallback_threshold=FALLBACK_THRESHOLD
            )
        self.hybrid_router = HybridRouter(
            self.embedding_router,
            self.llm_router,
            threshold=self.threshold
        )

    def confidence_gate(self, confidence: float) -> str:
        return ACTION_AUTO_ROUTE if confidence >= self.threshold else ACTION_HITL

    def route(self, query_id: int, query: str, strategy: str = DEFAULT_STRATEGY) -> FinalResult:
        norm_q = preprocess_query(query)

        candidate_tools = get_candidate_tools_v1(self.unified_pool)
        if not candidate_tools:
            candidate_tools = self.tools_all[:]  # fallback: all tools

        # Strategy dispatch
        if strategy == "Embedding_Based":
            decision = self.embedding_router.route(norm_q, candidate_tools)
        elif strategy == "LLM_Router":
            decision = self.llm_router.route(norm_q, candidate_tools)
        else:
            decision = self.hybrid_router.route(norm_q, candidate_tools)

        action = self.confidence_gate(decision.confidence)

        return FinalResult(
            query_id=query_id,
            query=query,
            strategy=strategy,
            decision=decision,
            action=action,
            threshold=self.threshold,
        )


D) Standard evaluation
* 1) Orchestration instantiation
* 2) Output Normalizer
* 3) Standard Experiment Set-
* 4) Run Standard Experiment
* 5) Evaluation


In [ ]:
# ============================
# D1) Orchestrator Instantiation (Version 1)
# ============================

orch_v1 = MiniOrchestratorV1_2(
    tool_pools=TOOL_POOLS,
    threshold=CONFIDENCE_THRESHOLD
)

print("Orch_v1 ready:", [p.name for p in TOOL_POOLS])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Orch_v1 ready: ['AllTools']


In [ ]:
# ============================
# D2) Output Normalizer V1
# ============================

def to_evaluation_format(out):
  return {
      "query_id": out.query_id,
      "query": out.query,
      "strategy": out.strategy,
      "selected_pool": out.decision.trace.get("pool", "unified"),
      "selected_tool": out.decision.selected_tool,
      "final_confidence": out.decision.confidence,
      "action": out.action,
      "trace": out.decision.trace,
  }

E) Experiments & Testing

In [ ]:
# ================================
# D3) Standard Experiment Setup
# ================================

# 1) Datasets
DATASETS = {
    "baseline": df_baseline.to_dict("records"),
    "noisy": df_noisy.to_dict("records"),
    "paraphrased": df_paraphrased.to_dict("records"),
    "translated": df_translated.to_dict("records"),
}

# 2) Strategies
STRATEGIES = [
    "Embedding_Based",
    "LLM_Router",
    "Hybrid"
]

# 3) Orchestrator
ORCH = orch_v1

# 4) Output settings
EXPORT_PATH = "experiment_results.csv"

# 5) Debug flags
PRINT_TRACE = False

# 6) Expected mapping
EXPECTED_MAP = {
    row["id"]: row["expected_tool"]
    for row in df_baseline.to_dict("records")
    }


In [ ]:
# ================================
# D4) Run Standard Experiment
# ================================

all_rows = []

for dataset_name, dataset in DATASETS.items():
    print (f"Running dataset: {dataset_name}")

    for strategy in STRATEGIES:
        print (f"Running strategy: {strategy}")

        for q in dataset:
            out = ORCH.route(
                query_id=q["id"],
                query=q["query"],
                strategy=strategy
            )

            row = to_evaluation_format(out)
            row["dataset"] = dataset_name
            all_rows.append(row)

df_results_v1 = pd.DataFrame(all_rows)

print("Done.")
print("Rows:", len(df_results_v1))
print(df_results_v1.head)

#export
df_results_v1.to_csv("df_results_v1.csv", index=False)
print("Saved df_results_v1")

Running dataset: baseline
Running strategy: Embedding_Based
Running strategy: LLM_Router


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Running strategy: Hybrid
Running dataset: noisy
Running strategy: Embedding_Based
Running strategy: LLM_Router
Running strategy: Hybrid
Running dataset: paraphrased
Running strategy: Embedding_Based
Running strategy: LLM_Router
Running strategy: Hybrid
Running dataset: translated
Running strategy: Embedding_Based
Running strategy: LLM_Router
Running strategy: Hybrid
Done.
Rows: 1950
<bound method NDFrame.head of       query_id                                              query  \
0            1     What are the distinct outline agreement types?   
1            2  Who is the Vendor for outline agreement 456781...   
2            3                  What material did 0001234567 buy?   
3            4                         What material is 10012345?   
4            5                               Who bought 10054321?   
...        ...                                                ...   
1945        49                                        用3句子解释供需原则.   
1946        50             Quell

In [ ]:
from google.colab import files
files.download("df_results_v1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ================================
# D5) Evaluation of the results
# ================================

# Architecture label
ARCHITECTURE = "unified"

# 1) Map expected tool
df_results_v1 = df_results_v1.copy()
df_results_v1["architecture"] = ARCHITECTURE
df_results_v1["expected_tool"] = df_results_v1["query_id"].map(EXPECTED_MAP)

# 2) Compute correctnes
df_results_v1["is_correct"] = df_results_v1["selected_tool"] == df_results_v1["expected_tool"]

# 3) Summary metrics
summary_v1 = (
    df_results_v1
    .groupby(["architecture", "dataset", "strategy"])
    .agg(
        accuracy=("is_correct", "mean"),
        confidence=("final_confidence", "mean"),
        auto_rate=("action", lambda x: (x == "ACTION_AUTO_ROUTE").mean()),
    )
    .reset_index()
)

print("Evaluation complete.")
print("Rows:", len(df_results_v1))
summary_v1

Evaluation complete.
Rows: 1950


,architecture,dataset,strategy,accuracy,confidence,auto_rate
0,unified,baseline,Embedding_Based,0.440,0.567059,0.0
1,unified,baseline,Hybrid,0.140,0.421803,0.0
2,unified,baseline,LLM_Router,0.140,0.421803,0.0
3,unified,noisy,Embedding_Based,0.360,0.545080,0.0
4,unified,noisy,Hybrid,0.130,0.434316,0.0
5,unified,noisy,LLM_Router,0.130,0.434316,0.0
6,unified,paraphrased,Embedding_Based,0.405,0.558568,0.0
7,unified,paraphrased,Hybrid,0.135,0.411992,0.0
8,unified,paraphrased,LLM_Router,0.135,0.411992,0.0
9,unified,translated,Embedding_Based,0.250,0.523892,0.0


E) Additional experiments

In [ ]:
# ================================
# E1) Additonal experiments
# ================================


EXPERIMENT_CONFIGS = [
    {"name": "thr_0.8_no_fallback", "threshold": 0.8, "use_fallback": False},
    {"name": "thr_0.7_no_fallback", "threshold": 0.7, "use_fallback": False},
    {"name": "thr_0.6_no_fallback", "threshold": 0.6, "use_fallback": False},
    {"name": "thr_0.8_fallback", "threshold": 0.8, "use_fallback": True},
]

all_rows = []

for config in EXPERIMENT_CONFIGS:

    print(f"\n=== Running experiment with config: {config['name']} ====")

    #Rebuild routers with config
    embedding_router = EmbeddingRouter(
        model_name=EMBEDDING_MODEL_NAME,
        use_fallback=config["use_fallback"],
        fallback_threshold=FALLBACK_THRESHOLD
        )
    llm_router = LLMRouter(
        model_name=LLM_MODEL_NAME,
        use_fallback=config["use_fallback"],
        fallback_threshold=FALLBACK_THRESHOLD
        )
    hybrid_router = HybridRouter(
        embedding_router=embedding_router,
        llm_router=llm_router,
        threshold=config["threshold"]
        )

    #Rebuild orchestrator
    orch = MiniOrchestratorV1_2(
        tool_pools=TOOL_POOLS,
        threshold=config["threshold"],
    )

    # Run experiment
    for dataset_name, dataset in DATASETS.items():
        print(f"Running dataset: {dataset}")
        for strategy in STRATEGIES:
            print(f"Running strategy: {strategy}")

            for q in dataset:
                out = orch.route(
                    query_id=q["id"],
                    query=q["query"],
                    strategy=strategy
                )
                row = to_evaluation_format(out)

                # Add experiment meta data
                row["dataset"] = dataset_name
                row["experiment_name"] = config["name"]
                row["threshold"] = config["threshold"]
                row["fallback_enabled"] = config["use_fallback"]

                all_rows.append(row)

df_experiments_v1 = pd.DataFrame(all_rows)

print("Done.")
print("Rows:", len(df_experiments_v1))
print(df_experiments_v1.head())

#export
df_experiments_v1.to_csv("df_experiments_v1.csv")
print("Saved df_experiments_v1")


=== Running experiment with config: thr_0.8_no_fallback ====


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Running dataset: [{'id': 1, 'query': 'What are the distinct outline agreement types?', 'expected_tool': 'oa_data_tool'}, {'id': 2, 'query': 'Who is the Vendor for outline agreement 456781234?', 'expected_tool': 'oa_data_tool'}, {'id': 3, 'query': 'What material did 0001234567 buy?', 'expected_tool': 'oa_data_tool'}, {'id': 4, 'query': 'What material is 10012345?', 'expected_tool': 'oa_data_tool'}, {'id': 5, 'query': 'Who bought 10054321?', 'expected_tool': 'oa_data_tool'}, {'id': 6, 'query': 'Please give me the company code related to 0000001234?', 'expected_tool': 'oa_data_tool'}, {'id': 7, 'query': "What is the total released value for all positions for vendor '0000009876'?", 'expected_tool': 'oa_data_fulfillment_tool'}, {'id': 8, 'query': "What is the released value for all positions for vendor '0000005432'?", 'expected_tool': 'oa_data_fulfillment_tool'}, {'id': 9, 'query': "What is the total quantity ordered for vendor '0000005432'?", 'expected_tool': 'oa_data_fulfillment_tool'}, {

In [ ]:
from google.colab import files
files.download("df_experiments_v1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ========================================
# E2) Evaluation of the experiment results
# ========================================

# Create a copy:
summary_exp_v1 = df_experiments_v1.copy()

# Create unified confidence column
if "final_confidence" in summary_exp_v1.columns:
    pass
elif "tool_confidence" in summary_exp_v1.columns:
    summary_exp_v1["final_confidence"] = summary_exp_v1["tool_confidence"]
elif "confidence" in summary_exp_v1.columns:
    summary_exp_v1["final_confidence"] = summary_exp_v1["confidence"]
else:
    summary_exp_v1["final_confidence"] = np.nan

summary_exp_v1["final_confidence"] = pd.to_numeric(summary_exp_v1["final_confidence"], errors="coerce")

# Create corecctness column if not already present
summary_exp_v1["expected_tool"] = summary_exp_v1["query_id"].map(EXPECTED_MAP)

if "is_correct" not in summary_exp_v1.columns:
    summary_exp_v1["is_correct"] = (
        summary_exp_v1["selected_tool"] == summary_exp_v1["expected_tool"]
    ).astype(float)

#Make fallback label better
summary_exp_v1["fallback_status"] = summary_exp_v1["fallback_enabled"].map({
    True: "With fallback",
    False: "No fallback"
})

# Build summary table
summary_exp_v1["architecture"] = "unified"
summary_exp_v1["is_auto_route"] =(
    summary_exp_v1["action"].astype(str).str.lower().isin(["auto_route", "auto route"])
).astype(float)

summary_exp_v1 = (
    summary_exp_v1
    .groupby(["architecture",
              "dataset",
              "strategy",
              "threshold",
              "fallback_status"
    ],
             dropna=False
    )
    .agg(
        accuracy=("is_correct", "mean"),
        avg_confidence=("final_confidence", "mean"),
        auto_rate=("is_auto_route", "mean"),
        n_queries=("query_id", "count")
    )
    .reset_index()
    )

# Round for display
summary_exp_v1["accuracy"] = summary_exp_v1["accuracy"].round(3)
summary_exp_v1["avg_confidence"] = summary_exp_v1["avg_confidence"].round(3)
summary_exp_v1["auto_rate"] = summary_exp_v1["auto_rate"].round(3)

# Sort nicely
summary_exp_v1 = summary_exp_v1.sort_values(
    by=["architecture", "threshold", "fallback_status", "dataset", "strategy"]
).reset_index(drop=True)

print("Evaluation complete:")
print("Rows:", len(summary_exp_v1))
summary_exp_v1

Evaluation complete:
Rows: 48


,architecture,dataset,strategy,threshold,fallback_status,accuracy,avg_confidence,auto_rate,n_queries
0,unified,baseline,Embedding_Based,0.6,No fallback,0.440,0.567,0.360,50
1,unified,baseline,Hybrid,0.6,No fallback,0.320,0.502,0.400,50
2,unified,baseline,LLM_Router,0.6,No fallback,0.140,0.422,0.060,50
3,unified,noisy,Embedding_Based,0.6,No fallback,0.360,0.545,0.185,200
4,unified,noisy,Hybrid,0.6,No fallback,0.200,0.471,0.235,200
5,unified,noisy,LLM_Router,0.6,No fallback,0.130,0.434,0.070,200
6,unified,paraphrased,Embedding_Based,0.6,No fallback,0.405,0.559,0.245,200
7,unified,paraphrased,Hybrid,0.6,No fallback,0.260,0.468,0.280,200
8,unified,paraphrased,LLM_Router,0.6,No fallback,0.135,0.412,0.045,200
9,unified,translated,Embedding_Based,0.6,No fallback,0.250,0.524,0.050,200


In [ ]:
from google.colab import files
files.download("df_experiments_v1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>